# 40 PCA20 Ensemble - Estructura Unificada + Nested CV 5x3

Plantilla robusta unificada para comparacion metodologica consistente.


## 1) Imports

In [1]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from scipy.stats import pearsonr

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = str(exc)

## 2) Configuración global

In [2]:
# =========================
# Identidad del notebook
# =========================
NOTEBOOK_TITLE = '40 PCA20 Ensemble - Estructura Unificada + Nested CV 5x3'
REPRESENTATION = 'pca20'  # pca20 | pca50 | snp_direct
MODELS_TO_RUN = ['random_forest', 'gradient_boosting', 'adaboost']

# =========================
# CV unificada (Nested 5x3)
# =========================
OUTER_SPLITS = 5
INNER_SPLITS = 3
RANDOM_STATE = 42

# =========================
# Parametros del grid (visibles al inicio)
# =========================
PARAM_GRIDS = {
    'ridge': {
        'model__alpha': [0.1, 1.0, 10.0],
    },
    'elasticnet': {
        'model__alpha': [0.01, 0.1, 1.0],
        'model__l1_ratio': [0.05, 0.1, 0.3],
    },
    'xgboost': {
        'model__n_estimators': [300, 500],
        'model__learning_rate': [0.03, 0.05],
        'model__max_depth': [3, 5],
        'model__subsample': [0.7, 1.0],
        'model__colsample_bytree': [0.1, 0.3],
        'model__reg_lambda': [1.0, 5.0],
    },
    'svr_linear': {
        'model__C': [0.1, 1, 10, 100],
        'model__epsilon': [0.01, 0.1, 0.5],
    },
    'svr_rbf': {
        'model__C': [0.1, 1, 10, 100],
        'model__epsilon': [0.01, 0.1, 0.5],
        'model__gamma': ['scale', 'auto', 0.01, 0.1],
    },
    'decision_tree': {
        'model__max_depth': [3, 5, 10, 20, None],
        'model__min_samples_split': [2, 5, 10, 20],
        'model__min_samples_leaf': [1, 2, 5, 10],
        'model__max_features': [None, 'sqrt', 'log2'],
        'model__ccp_alpha': [0.0, 0.001, 0.01, 0.1],
    },
    'random_forest': {
        'model__n_estimators': [300, 500],
        'model__max_depth': [15, 25, None],
        'model__min_samples_leaf': [1, 2, 3],
        'model__max_features': [0.3, 0.5, 'sqrt'],
    },
    'gradient_boosting': {
        'model__n_estimators': [300, 500],
        'model__learning_rate': [0.005, 0.01, 0.03],
        'model__max_depth': [3, 4, 5],
        'model__min_samples_leaf': [1, 2],
        'model__subsample': [0.7, 1.0],
    },
    'adaboost': {
        'estimator_max_depth': [2, 3],
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1, 0.2],
        'model__loss': ['linear', 'square'],
    },
}

# =========================
# Traits candidatos
# =========================
TRAITS_CANDIDATOS = [
    'blumeria_graminis',
    'heading_date',
    'lodging',
    'plant_height',
    'puccinia_hordei',
    'ramularia_collo_cygni',
    'rhynchosporium',
]

# =========================
# Rutas de entrada
# =========================
PHENO_REL = 'pheno_data/BLUEs_across_env.tsv'
META_REL = 'pheno_data/giae121_supplemental_file.xlsx'
FEATURE_REL = 'pca20_mind0.20_g0.10_nomaf_pruned.eigenvec'
N_PCS = 20

# =========================
# Salida
# =========================
OUTPUT_BASE = Path('outputs_nested5x3')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# Checks robustos de configuracion
# =========================
if not MODELS_TO_RUN:
    raise ValueError('MODELS_TO_RUN esta vacio')

missing_grid_models = [m for m in MODELS_TO_RUN if m not in PARAM_GRIDS]
if missing_grid_models:
    raise ValueError(f'Modelos sin grid definido en PARAM_GRIDS: {missing_grid_models}')

for m in MODELS_TO_RUN:
    g = PARAM_GRIDS[m]
    if not isinstance(g, dict) or len(g) == 0:
        raise ValueError(f'Grid invalido para modelo={m}: {g}')

if REPRESENTATION in {'pca20', 'pca50'} and (N_PCS is None or N_PCS < 1):
    raise ValueError('N_PCS debe ser >=1 para representaciones PCA')

if REPRESENTATION == 'snp_direct' and not FEATURE_REL.lower().endswith('.raw'):
    raise ValueError("Para snp_direct, FEATURE_REL debe apuntar a un .raw")

if not XGBOOST_AVAILABLE and 'xgboost' in [m.lower() for m in MODELS_TO_RUN]:
    print(f"[WARN] XGBoost no disponible: {XGBOOST_IMPORT_ERROR}")


def _grid_size(grid_dict):
    sizes = [len(v) for v in grid_dict.values() if isinstance(v, list)]
    out = 1
    for s in sizes:
        out *= s
    return out

print(f'NOTEBOOK_TITLE: {NOTEBOOK_TITLE}')
print(f'REPRESENTATION: {REPRESENTATION}')
print(f'MODELS_TO_RUN: {MODELS_TO_RUN}')
print(f'Nested CV: outer={OUTER_SPLITS}, inner={INNER_SPLITS}, random_state={RANDOM_STATE}')
for m in MODELS_TO_RUN:
    print(f'Grid {m}: {_grid_size(PARAM_GRIDS[m])} combinaciones')


NOTEBOOK_TITLE: 40 PCA20 Ensemble - Estructura Unificada + Nested CV 5x3
REPRESENTATION: pca20
MODELS_TO_RUN: ['random_forest', 'gradient_boosting', 'adaboost']
Nested CV: outer=5, inner=3, random_state=42
Grid random_forest: 54 combinaciones
Grid gradient_boosting: 72 combinaciones
Grid adaboost: 24 combinaciones


## 3) Utilidades

In [3]:
def resolve_path(rel_path: str) -> Path:
    # Resuelve una ruta buscando en cwd y padres para robustez
    rel = Path(rel_path)
    candidates = [
        Path.cwd() / rel,
        Path.cwd().parent / rel,
        Path.cwd().parent.parent / rel,
        Path.cwd().parent.parent.parent / rel,
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"No se encontró: {rel_path}. Candidatos: {[str(x) for x in candidates]}")


def normalize_name(x) -> str | None:
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s or None


def safe_pearson(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan, np.nan
    if np.nanstd(y_true) == 0 or np.nanstd(y_pred) == 0:
        return np.nan, np.nan
    try:
        return pearsonr(y_true, y_pred)
    except Exception:
        return np.nan, np.nan


def metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    r, p = safe_pearson(y_true, y_pred)
    return {
        "pearson_r": float(r) if pd.notna(r) else np.nan,
        "pearson_p": float(p) if pd.notna(p) else np.nan,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def to_serializable_params(params: dict) -> str:
    clean = {}
    for k, v in params.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        else:
            clean[k] = repr(v)
    return json.dumps(clean, ensure_ascii=False, sort_keys=True)

## 4) Carga y preparación de datos

In [4]:
# =========================
# Carga de datos
# =========================
PHENO_PATH = resolve_path(PHENO_REL)
META_PATH = resolve_path(META_REL)
FEATURE_PATH = resolve_path(FEATURE_REL)

print(f"PHENO_PATH: {PHENO_PATH}")
print(f"META_PATH: {META_PATH}")
print(f"FEATURE_PATH: {FEATURE_PATH}")


def load_trait_table(pheno_path: Path, meta_path: Path, traits: list[str]) -> pd.DataFrame:
    ph = pd.read_csv(pheno_path, sep="	")
    meta = pd.read_excel(meta_path, header=2)

    if "genotypes" not in ph.columns:
        raise ValueError("No existe columna 'genotypes' en el fichero fenotípico.")

    meta_cols_lower = {str(c).strip().lower(): c for c in meta.columns}
    if "genotype" not in meta_cols_lower:
        raise ValueError("No existe columna 'genotype' en metadata.")
    genotype_col = meta_cols_lower["genotype"]

    biosample_candidates = [c for c in meta.columns if "biosamples" in str(c).lower() and "id" in str(c).lower()]
    if not biosample_candidates:
        raise ValueError("No se encontró columna BioSamples ID en metadata.")
    biosample_col = biosample_candidates[0]

    traits_in_pheno = [t for t in traits if t in ph.columns]
    if not traits_in_pheno:
        raise ValueError("No hay traits candidatos en BLUEs_across_env.tsv")

    ph = ph[["genotypes"] + traits_in_pheno].copy()
    ph["GENO_NORM"] = ph["genotypes"].map(normalize_name)
    ph_agg = ph.groupby("GENO_NORM", as_index=False)[traits_in_pheno].mean(numeric_only=True)

    meta2 = meta[[genotype_col, biosample_col]].copy()
    meta2["GENO_NORM"] = meta2[genotype_col].map(normalize_name)
    meta2["IID"] = meta2[biosample_col].astype(str).str.strip()
    meta2 = meta2.dropna(subset=["GENO_NORM", "IID"]).drop_duplicates(subset=["GENO_NORM"])

    trait_table = ph_agg.merge(meta2[["GENO_NORM", "IID"]], on="GENO_NORM", how="inner")
    return trait_table


def load_feature_table(representation: str, feature_path: Path, n_pcs: int | None = None):
    representation = representation.lower()

    if representation in {"pca20", "pca50"}:
        if not n_pcs or n_pcs < 1:
            raise ValueError("n_pcs debe ser >=1 para PCA")

        feat = pd.read_csv(feature_path, sep=r"\s+", header=None)

        # Soporta ambos formatos habituales de .eigenvec:
        # 1) FID, IID, PC1..PCn  -> n_pcs + 2 columnas
        # 2) IID, PC1..PCn       -> n_pcs + 1 columnas
        if feat.shape[1] >= n_pcs + 2:
            cols = ["FID", "IID"] + [f"PC{i}" for i in range(1, n_pcs + 1)]
            feat = feat.iloc[:, :len(cols)].copy()
            feat.columns = cols
        elif feat.shape[1] >= n_pcs + 1:
            cols = ["IID"] + [f"PC{i}" for i in range(1, n_pcs + 1)]
            feat = feat.iloc[:, :len(cols)].copy()
            feat.columns = cols
        else:
            raise ValueError(
                f"PCA con columnas insuficientes: esperadas >= {n_pcs + 1}, encontradas {feat.shape[1]}"
            )

        feature_cols = [c for c in feat.columns if c.startswith("PC")]
        return feat[["IID"] + feature_cols].copy(), feature_cols

    if representation == "snp_direct":
        feat = pd.read_csv(feature_path, sep=r"\s+")
        base_cols = {"FID", "IID", "PAT", "MAT", "SEX", "PHENOTYPE"}
        feature_cols = [c for c in feat.columns if c not in base_cols]
        if not feature_cols:
            raise ValueError("No se detectaron columnas SNP en el fichero .raw")
        return feat[["IID"] + feature_cols].copy(), feature_cols

    raise ValueError(f"Representación no soportada: {representation}")

trait_table = load_trait_table(PHENO_PATH, META_PATH, TRAITS_CANDIDATOS)
feature_table, feature_cols = load_feature_table(REPRESENTATION, FEATURE_PATH, N_PCS)

model_df = feature_table.merge(trait_table, on="IID", how="inner")
traits_disponibles = [t for t in TRAITS_CANDIDATOS if t in model_df.columns]

print(f"feature_table: {feature_table.shape}")
print(f"trait_table: {trait_table.shape}")
print(f"model_df: {model_df.shape}")
print(f"n_features: {len(feature_cols)}")
print(f"traits_disponibles: {traits_disponibles}")

if not traits_disponibles:
    raise ValueError("No hay traits disponibles tras merge")

PHENO_PATH: c:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados\pheno_data\BLUEs_across_env.tsv
META_PATH: c:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados\pheno_data\giae121_supplemental_file.xlsx
FEATURE_PATH: c:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\pca20_mind0.20_g0.10_nomaf_pruned.eigenvec
feature_table: (1111, 21)
trait_table: (1110, 9)
model_df: (1110, 29)
n_features: 20
traits_disponibles: ['blumeria_graminis', 'heading_date', 'lodging', 'plant_height', 'puccinia_hordei', 'ramularia_collo_cygni', 'rhynchosporium']


## 5) Especificación de modelos y grids

In [5]:
# =========================
# Especificaciones de modelos
# =========================
def get_model_spec(model_name: str, random_state: int = RANDOM_STATE):
    m = model_name.lower()

    if m not in PARAM_GRIDS:
        raise ValueError(f'Modelo no soportado o sin grid: {model_name}')

    if m == 'ridge':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', Ridge(random_state=random_state)),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'elasticnet':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', ElasticNet(max_iter=20000, random_state=random_state)),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'xgboost':
        if not XGBOOST_AVAILABLE:
            raise ImportError('XGBoost no disponible en este entorno')
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            (
                'model',
                XGBRegressor(
                    objective='reg:squarederror',
                    tree_method='hist',
                    random_state=random_state,
                    n_jobs=1,
                    verbosity=0,
                ),
            ),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'svr_linear':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', SVR(kernel='linear')),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'svr_rbf':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', SVR(kernel='rbf')),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'decision_tree':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', DecisionTreeRegressor(random_state=random_state)),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'random_forest':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', RandomForestRegressor(random_state=random_state, n_jobs=-1)),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'gradient_boosting':
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', GradientBoostingRegressor(random_state=random_state)),
        ])
        return pipe, PARAM_GRIDS[m]

    if m == 'adaboost':
        ada = AdaBoostRegressor(random_state=random_state)
        estimator_param = 'model__estimator' if 'estimator' in ada.get_params() else 'model__base_estimator'

        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', ada),
        ])

        base_grid = PARAM_GRIDS[m]
        grid = {
            estimator_param: [
                DecisionTreeRegressor(max_depth=d, random_state=random_state)
                for d in base_grid['estimator_max_depth']
            ],
            'model__n_estimators': base_grid['model__n_estimators'],
            'model__learning_rate': base_grid['model__learning_rate'],
            'model__loss': base_grid['model__loss'],
        }
        return pipe, grid

    raise ValueError(f'Modelo no soportado: {model_name}')


## 6) Nested CV 5x3

In [6]:
# =========================
# Nested CV (5x3) y evaluación OOF externa
# =========================
def run_nested_cv_trait(model_df: pd.DataFrame, feature_cols: list[str], trait: str, model_name: str):
    sub = model_df[feature_cols + [trait]].dropna().copy()
    X = sub[feature_cols].to_numpy()
    y = sub[trait].to_numpy(dtype=float)

    if X.shape[0] < OUTER_SPLITS:
        raise ValueError(f"Trait={trait}: n={X.shape[0]} < OUTER_SPLITS={OUTER_SPLITS}")

    outer_cv = KFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    base_pipe, param_grid = get_model_spec(model_name, random_state=RANDOM_STATE)

    oof_pred = np.full(y.shape[0], np.nan, dtype=float)
    fold_rows = []
    best_params_outer = []

    t0 = time.perf_counter()

    for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
        grid = GridSearchCV(
            estimator=clone(base_pipe),
            param_grid=param_grid,
            scoring="r2",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            error_score="raise",
        )
        grid.fit(X[tr_idx], y[tr_idx])
        pred = grid.best_estimator_.predict(X[te_idx])
        oof_pred[te_idx] = pred

        m = metrics_dict(y[te_idx], pred)
        fold_rows.append(
            {
                "trait": trait,
                "model": model_name,
                "fold": fold_id,
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "pearson_r": m["pearson_r"],
                "rmse": m["rmse"],
                "mae": m["mae"],
                "r2": m["r2"],
                "best_params": to_serializable_params(grid.best_params_),
                "best_inner_r2": float(grid.best_score_),
            }
        )
        best_params_outer.append(to_serializable_params(grid.best_params_))

    elapsed = time.perf_counter() - t0

    if np.isnan(oof_pred).any():
        raise RuntimeError(f"Trait={trait}, model={model_name}: NaN en OOF externo")

    g = metrics_dict(y, oof_pred)
    mode_params = Counter(best_params_outer).most_common(1)[0][0] if best_params_outer else "{}"

    summary_row = {
        "representation": REPRESENTATION,
        "trait": trait,
        "model": model_name,
        "n_individuos": int(len(y)),
        "n_features": int(X.shape[1]),
        "cv_outer": f"KFold({OUTER_SPLITS}, shuffle=True, random_state={RANDOM_STATE})",
        "cv_inner": f"KFold({INNER_SPLITS}, shuffle=True, random_state={RANDOM_STATE})",
        "pearson_r": g["pearson_r"],
        "pearson_p": g["pearson_p"],
        "rmse": g["rmse"],
        "mae": g["mae"],
        "r2": g["r2"],
        "fit_time_s": float(elapsed),
        "best_params_mode_outer": mode_params,
    }

    pred_df = pd.DataFrame({"trait": trait, "model": model_name, "y_true": y, "y_pred_oof_outer": oof_pred})
    folds_df = pd.DataFrame(fold_rows)

    return summary_row, folds_df, pred_df

## 7) Ejecución

In [7]:
# =========================
# Ejecución por bloques de modelo
# =========================
run_dir = OUTPUT_BASE / Path(NOTEBOOK_TITLE.lower().replace(" ", "_").replace("/", "_"))
run_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
folds_parts = []
preds_parts = []
errors = []

for model_name in MODELS_TO_RUN:
    print("=" * 90)
    print(f"MODELO: {model_name}")
    print("=" * 90)

    for trait in traits_disponibles:
        print(f"  -> trait={trait}")
        try:
            srow, fdf, pdf = run_nested_cv_trait(model_df, feature_cols, trait, model_name)
            summary_rows.append(srow)
            folds_parts.append(fdf)
            preds_parts.append(pdf)
            print(f"     r2={srow['r2']:.4f} | pearson={srow['pearson_r']:.4f} | rmse={srow['rmse']:.4f}")
        except Exception as exc:
            msg = f"[ERROR] model={model_name}, trait={trait}: {exc}"
            print(msg)
            errors.append({"model": model_name, "trait": trait, "error": str(exc)})

summary_df = pd.DataFrame(summary_rows)
folds_df = pd.concat(folds_parts, ignore_index=True) if folds_parts else pd.DataFrame()
preds_df = pd.concat(preds_parts, ignore_index=True) if preds_parts else pd.DataFrame()
errors_df = pd.DataFrame(errors)

summary_csv = run_dir / "summary_nested5x3.csv"
folds_csv = run_dir / "fold_metrics_nested5x3.csv"
preds_csv = run_dir / "predictions_oof_outer_nested5x3.csv"
errors_csv = run_dir / "errors_nested5x3.csv"

summary_df.to_csv(summary_csv, index=False)
folds_df.to_csv(folds_csv, index=False)
preds_df.to_csv(preds_csv, index=False)
errors_df.to_csv(errors_csv, index=False)

print("\n[OK] Archivos guardados:")
print(f"- {summary_csv}")
print(f"- {folds_csv}")
print(f"- {preds_csv}")
print(f"- {errors_csv}")

if not summary_df.empty:
    display(summary_df.sort_values(["trait", "r2"], ascending=[True, False]).reset_index(drop=True))
else:
    print("[WARN] summary_df vacío. Revisar errors_df")


MODELO: random_forest
  -> trait=blumeria_graminis
     r2=0.5241 | pearson=0.7241 | rmse=0.8029
  -> trait=heading_date
     r2=0.6203 | pearson=0.7883 | rmse=19.5805
  -> trait=lodging
     r2=0.7610 | pearson=0.8726 | rmse=0.7738
  -> trait=plant_height
     r2=0.6291 | pearson=0.7936 | rmse=8.7282
  -> trait=puccinia_hordei
     r2=0.4893 | pearson=0.6995 | rmse=0.8494
  -> trait=ramularia_collo_cygni
     r2=0.1130 | pearson=0.3461 | rmse=1.0557
  -> trait=rhynchosporium
     r2=0.0189 | pearson=0.2103 | rmse=0.9066
MODELO: gradient_boosting
  -> trait=blumeria_graminis
     r2=0.5200 | pearson=0.7215 | rmse=0.8064
  -> trait=heading_date
     r2=0.6139 | pearson=0.7835 | rmse=19.7463
  -> trait=lodging
     r2=0.7543 | pearson=0.8686 | rmse=0.7846
  -> trait=plant_height
     r2=0.6220 | pearson=0.7887 | rmse=8.8106
  -> trait=puccinia_hordei
     r2=0.4761 | pearson=0.6902 | rmse=0.8604
  -> trait=ramularia_collo_cygni
     r2=0.1116 | pearson=0.3343 | rmse=1.0566
  -> trait=rhy

,representation,trait,model,n_individuos,n_features,cv_outer,cv_inner,pearson_r,pearson_p,rmse,mae,r2,fit_time_s,best_params_mode_outer
0,pca20,blumeria_graminis,random_forest,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.724054,6.096108e-181,0.802930,0.619455,0.524128,548.575134,"{""model__max_depth"": 25, ""model__max_features""..."
1,pca20,blumeria_graminis,gradient_boosting,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.721474,4.626874e-179,0.806411,0.626301,0.519993,1600.718683,"{""model__learning_rate"": 0.01, ""model__max_dep..."
2,pca20,blumeria_graminis,adaboost,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.713558,2.012602e-173,0.818579,0.640545,0.505398,105.455961,"{""model__estimator"": ""DecisionTreeRegressor(ma..."
3,pca20,heading_date,random_forest,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.788280,6.370693e-236,19.580469,12.650953,0.620310,944.823277,"{""model__max_depth"": null, ""model__max_feature..."
4,pca20,heading_date,gradient_boosting,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.783500,3.419725e-231,19.746282,12.362323,0.613852,971.249147,"{""model__learning_rate"": 0.01, ""model__max_dep..."
5,pca20,heading_date,adaboost,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.742841,2.609447e-195,21.665242,16.731787,0.535153,89.485324,"{""model__estimator"": ""DecisionTreeRegressor(ma..."
6,pca20,lodging,random_forest,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.872573,0.000000e+00,0.773775,0.595131,0.761035,950.093090,"{""model__max_depth"": null, ""model__max_feature..."
7,pca20,lodging,gradient_boosting,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.868555,0.000000e+00,0.784577,0.600593,0.754316,978.772695,"{""model__learning_rate"": 0.01, ""model__max_dep..."
8,pca20,lodging,adaboost,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.853784,2.291835e-316,0.833648,0.647582,0.722623,75.326359,"{""model__estimator"": ""DecisionTreeRegressor(ma..."
9,pca20,plant_height,random_forest,1110,20,"KFold(5, shuffle=True, random_state=42)","KFold(3, shuffle=True, random_state=42)",0.793593,2.513150e-241,8.728224,6.530884,0.629066,713.921579,"{""model__max_depth"": 15, ""model__max_features""..."


## Notas metodológicas

- Estructura unificada tipo `53`.
- Validación cruzada homogénea: **Nested CV 5x3** para todos los modelos.
- Métricas comparables: `Pearson`, `RMSE`, `MAE`, `R2`.
- Predicciones reportadas: OOF del loop externo.
- Preprocesado en pipeline (imputación/escalado según modelo) para minimizar leakage.